# CHƯƠNG 1 – THU THẬP DỮ LIỆU
**Môn:** Nhập Môn Khoa Học Dữ Liệu  
**Chủ đề:** Sức khỏe tâm thần sinh viên đại học *(Student Mental Health)*  
**Bộ dữ liệu:** 50 cặp Q&A Anh–Việt · 5 nhóm nội dung · 5 luật sinh câu · ChatGPT + Gemini

## 0. Import thư viện

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)

print('Thu vien da tai thanh cong.')

---
## 1.1 – Bộ dữ liệu gốc 50 cặp Q&A tiếng Anh

Bộ dữ liệu được xây dựng xoay quanh chủ đề **sức khỏe tâm thần sinh viên đại học**, chia thành 5 nhóm:

| Nhóm | ID | Mô tả |
|------|-----|--------|
| Frequency | Q01–Q10 | Tần suất trải nghiệm các vấn đề tâm lý |
| Purpose   | Q11–Q20 | Mục đích tìm kiếm hỗ trợ sức khỏe tâm thần |
| Benefit   | Q21–Q30 | Lợi ích của việc chăm sóc sức khỏe tâm thần |
| Concerns  | Q31–Q40 | Lo ngại và rủi ro liên quan |
| Ethics    | Q41–Q50 | Đạo đức trong hỗ trợ tâm thần tại trường |

In [ ]:
df_data = pd.read_csv('Data.csv')

print(f'Tong so cap Q&A: {len(df_data)}')
print(f'So cot: {df_data.shape[1]}')
print(f'\nPhan bo theo nhom:')
print(df_data['Group'].value_counts())

df_data[['ID','Group','Question_EN','Answer_EN']].head(10)

---
## 1.2 – Dịch và đánh giá câu dịch

Toàn bộ 50 cặp Q&A được dịch sang tiếng Việt bằng **Gemini** với prompt:

> *"Bạn là chuyên gia dịch thuật Anh–Việt học thuật. Hãy dịch các câu hỏi và câu trả lời sau sang tiếng Việt một cách tự nhiên, đúng nghĩa và phù hợp với ngữ cảnh sinh viên đại học."*

Câu dịch được đánh giá theo **3 tiêu chí** (thang 1–5):
- **Fluency** – Độ trôi chảy, tự nhiên khi đọc  
- **Adequacy** – Đầy đủ nghĩa, không mất ý so với bản gốc  
- **Grammar** – Đúng ngữ pháp tiếng Việt

In [ ]:
print('=== Bang song ngu + danh gia (5 cap dau) ===')
print(df_data[['ID','Question_EN','Question_VI_Gemini','Fluency','Adequacy','Grammar']].head(5).to_string(index=False))

print('\n=== Thong ke diem danh gia dich thuat ===')
print(df_data[['Fluency','Adequacy','Grammar']].describe().round(2))

### 1.2.1 – Tính điểm BLEU-1 đến BLEU-4

BLEU *(Bilingual Evaluation Understudy)* đo mức độ trùng khớp n-gram giữa câu hypothesis và câu reference.  
Trong bài này BLEU được tính trên **hai hướng**:
1. **Câu VI vs câu EN gốc** – đo mức tương đồng token cross-lingual theo yêu cầu đề bài  
2. **Câu sinh (EN) vs câu gốc (EN)** – đo mức độ biến đổi của 5 luật sinh câu

In [ ]:
def compute_bleu(reference: str, hypothesis: str):
    """Tinh BLEU-1 den BLEU-4 cho 1 cap cau."""
    ref_tokens = reference.lower().split()
    hyp_tokens = hypothesis.lower().split()
    sf = SmoothingFunction().method1
    b1 = sentence_bleu([ref_tokens], hyp_tokens, weights=(1,0,0,0),         smoothing_function=sf)
    b2 = sentence_bleu([ref_tokens], hyp_tokens, weights=(0.5,0.5,0,0),     smoothing_function=sf)
    b3 = sentence_bleu([ref_tokens], hyp_tokens, weights=(1/3,1/3,1/3,0),   smoothing_function=sf)
    b4 = sentence_bleu([ref_tokens], hyp_tokens, weights=(.25,.25,.25,.25), smoothing_function=sf)
    return round(b1,4), round(b2,4), round(b3,4), round(b4,4)

# Tinh BLEU: cau VI (hypothesis) vs cau EN (reference)
bleu_results = df_data.apply(
    lambda r: compute_bleu(r['Question_EN'], r['Question_VI_Gemini']), axis=1
)
df_data[['BLEU1','BLEU2','BLEU3','BLEU4']] = pd.DataFrame(bleu_results.tolist(), index=df_data.index)

# Bang tom tat
bleu_summary = pd.DataFrame({
    'Do do':    ['BLEU-1','BLEU-2','BLEU-3','BLEU-4'],
    'Gia tri TB': [
        df_data['BLEU1'].mean(), df_data['BLEU2'].mean(),
        df_data['BLEU3'].mean(), df_data['BLEU4'].mean()
    ],
    'Giai thich': [
        'Do trung khop 1-gram (tu don)',
        'Do trung khop cap tu lien tiep',
        'Ngu doan 3 tu duoc bao toan',
        'Ngu doan 4 tu – thap do khac cu phap Anh-Viet'
    ]
})
bleu_summary['Gia tri TB'] = bleu_summary['Gia tri TB'].round(3)
print('Bang 1.2: Diem BLEU trung binh cua bo du lieu dich thuat')
print(bleu_summary.to_string(index=False))

In [ ]:
# Ham tinh BLEU cho tung cau – version hien thi dep
def compute_bleu(reference, hypothesis):
    ref_tokens = reference.split()
    hyp_tokens = hypothesis.split()
    sf = SmoothingFunction().method1
    b1 = sentence_bleu([ref_tokens], hyp_tokens, weights=(1,0,0,0), smoothing_function=sf)
    b2 = sentence_bleu([ref_tokens], hyp_tokens, weights=(0.5,0.5,0,0), smoothing_function=sf)
    b3 = sentence_bleu([ref_tokens], hyp_tokens, weights=(1/3,1/3,1/3,0), smoothing_function=sf)
    b4 = sentence_bleu([ref_tokens], hyp_tokens, weights=(0.25,0.25,0.25,0.25), smoothing_function=sf)
    return b1, b2, b3, b4

# Ap dung (so sanh cau VI voi cau EN token hoa song song)
results = df_data.apply(lambda r: compute_bleu(r['Question_EN'], r['Question_VI_Gemini']), axis=1)
df_data[['BLEU1','BLEU2','BLEU3','BLEU4']] = pd.DataFrame(results.tolist())
print(df_data[['ID','BLEU1','BLEU2','BLEU3','BLEU4']].head(10))

In [ ]:
# Bieu do phan bo diem BLEU
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
colors = ['#4CAF50','#2196F3','#FF9800','#E91E63']

for i, (col, color) in enumerate(zip(['BLEU1','BLEU2','BLEU3','BLEU4'], colors)):
    mean_val = df_data[col].mean()
    axes[i].hist(df_data[col], bins=15, color=color, edgecolor='white', alpha=0.85)
    axes[i].axvline(mean_val, color='black', linestyle='--', linewidth=1.5)
    axes[i].set_title(f'BLEU-{i+1}\nMean = {mean_val:.3f}', fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Score', fontsize=10)
    axes[i].set_ylabel('Count', fontsize=10)

plt.suptitle('Phan bo diem BLEU-1 den BLEU-4 (50 cap Q&A dich thuat)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('bleu_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 1.3 – Sinh thêm dữ liệu bằng 5 Luật Ngôn Ngữ

**5 luật biến đổi ngôn ngữ** áp dụng cho từng câu gốc:

| Luật | Tên luật | Mô tả |
|------|----------|--------|
| **Rule 1** | Past Tense Shift | Chuyển sang thì quá khứ: *"In the past semester, ..."* / *"Looking back, ..."* |
| **Rule 2** | How-to Reframe | Đổi sang dạng hỏi cách: *"How do you [manage/cope]..."* |
| **Rule 3** | Synonym Substitution | Thay từ đồng nghĩa: *stress→anxiety, students→learners, support→assistance* |
| **Rule 4** | Perspective Shift | Góc nhìn nhóm: *"Among students, ..."* / *"It has been observed that..."* |
| **Rule 5** | Adverbial Addition | Thêm trạng ngữ: *"Generally speaking, ..."* / *"In most cases, ..."* |

Mỗi câu gốc → **5 câu mới** từ ChatGPT + **5 câu mới** từ Gemini → **250 câu/nguồn**.

In [ ]:
df_chatgpt = pd.read_csv('DChatGPT.csv')
df_gemini  = pd.read_csv('DGemini.csv')

print(f'DChatGPT.csv: {len(df_chatgpt)} cau | {df_chatgpt["Original_ID"].nunique()} IDs')
print(f'DGemini.csv:  {len(df_gemini)} cau  | {df_gemini["Original_ID"].nunique()} IDs')
print(f'\nPhan bo theo Rule (ChatGPT):')
print(df_chatgpt['Rule'].value_counts().sort_index())

# Vi du: Q01 – 5 bien the tu ChatGPT
orig_q = df_data[df_data['ID']=='Q01']['Question_EN'].values[0]
orig_a = df_data[df_data['ID']=='Q01']['Answer_EN'].values[0]
print(f'\n=== Vi du Q01 – cau goc ===')
print(f'Q: {orig_q}')
print(f'A: {orig_a}')
print(f'\n=== 5 bien the tu ChatGPT ===')
for _, row in df_chatgpt[df_chatgpt['Original_ID']=='Q01'].iterrows():
    print(f'[{row["Rule"]}] Q: {row["New_Question"]}')
    print(f'         A: {row["New_Answer"]}')
    print()

In [ ]:
# BLEU EN-EN: so sanh cau sinh vs cau goc de do muc do bien doi
orig_map = df_data.set_index('ID')[['Question_EN','Answer_EN']].to_dict('index')

def add_bleu_gen(dfgen):
    records = []
    for _, r in dfgen.iterrows():
        if r['Original_ID'] in orig_map:
            orig_q = orig_map[r['Original_ID']]['Question_EN']
            b1,b2,b3,b4 = compute_bleu(orig_q, r['New_Question'])
            records.append({
                'Original_ID': r['Original_ID'], 'Rule': r['Rule'], 'Source': r['Source'],
                'BLEU1':b1, 'BLEU2':b2, 'BLEU3':b3, 'BLEU4':b4
            })
    return pd.DataFrame(records)

bleu_cgpt = add_bleu_gen(df_chatgpt)
bleu_gem  = add_bleu_gen(df_gemini)

print('=== BLEU trung binh: cau sinh vs cau goc EN ===')
comp = pd.DataFrame({
    'ChatGPT': bleu_cgpt[['BLEU1','BLEU2','BLEU3','BLEU4']].mean(),
    'Gemini':  bleu_gem[['BLEU1','BLEU2','BLEU3','BLEU4']].mean()
}).round(3)
print(comp)

In [ ]:
# Bieu do BLEU theo tung Rule – ChatGPT vs Gemini
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()
rules = ['Rule 1','Rule 2','Rule 3','Rule 4','Rule 5']
bleu_metrics = ['BLEU1','BLEU2','BLEU3','BLEU4']

for idx, metric in enumerate(bleu_metrics):
    ax = axes[idx]
    cgpt_vals = bleu_cgpt.groupby('Rule')[metric].mean().reindex(rules)
    gem_vals  = bleu_gem.groupby('Rule')[metric].mean().reindex(rules)
    x = np.arange(len(rules))
    w = 0.35
    bars1 = ax.bar(x - w/2, cgpt_vals, w, label='ChatGPT', color='#1565C0', alpha=0.85)
    bars2 = ax.bar(x + w/2, gem_vals,  w, label='Gemini',  color='#2E7D32', alpha=0.85)
    for bars in [bars1, bars2]:
        for bar in bars:
            h = bar.get_height()
            if not np.isnan(h):
                ax.text(bar.get_x()+bar.get_width()/2, h+0.008,
                        f'{h:.2f}', ha='center', va='bottom', fontsize=8)
    ax.set_title(f'BLEU-{idx+1} theo Rule', fontsize=11, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels([r.replace(' ','\n') for r in rules], fontsize=9)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel('Score', fontsize=9)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Diem BLEU cau sinh (ChatGPT vs Gemini) theo tung Rule', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('bleu_by_rule.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 1.4 – Phiếu đánh giá dữ liệu sinh thêm

**Phương pháp:** Dùng Gemini với prompt:

> *"You are a linguistics quality evaluator. Rate the following generated question on 3 criteria and provide a short comment."*

**3 tiêu chí đánh giá:**

| Tiêu chí | Thang điểm | Mô tả |
|----------|-----------|--------|
| **Naturalness** | 1–5 | Câu đọc tự nhiên như người bản ngữ không? |
| **Meaning** | 1–5 | Bảo toàn ý nghĩa so với câu gốc không? |
| **Grammar** | 1–5 | Đúng ngữ pháp tiếng Anh không? |

In [ ]:
df_eval = pd.read_csv('EvaluationSheet.csv')

print(f'Tong so cau duoc danh gia: {len(df_eval)}')
print(f'Nguon: {df_eval["Source"].value_counts().to_dict()}')
print(f'So ID dai dien: {df_eval["ID"].nunique()} IDs')
print()
print('=== Thong ke diem theo nguon ===')
print(df_eval.groupby('Source')[['Naturalness','Meaning','Grammar']].mean().round(2))
print()
print('=== Phieu danh gia mau ===')
df_eval.head(10)

In [ ]:
# Bieu do 1: So sanh diem ChatGPT vs Gemini
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Grouped bar ---
ax1 = axes[0]
summary = df_eval.groupby('Source')[['Naturalness','Meaning','Grammar']].mean().round(2)
x = np.arange(3)
w = 0.35
clrs = ['#1565C0','#2E7D32']
for i, (src, clr) in enumerate(zip(summary.index, clrs)):
    bars = ax1.bar(x + i*w, summary.loc[src], w, label=src, color=clr, alpha=0.85)
    for bar in bars:
        ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.04,
                 f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax1.set_xticks(x + w/2)
ax1.set_xticklabels(['Naturalness','Meaning','Grammar'], fontsize=11)
ax1.set_ylim(0, 6)
ax1.set_ylabel('Diem trung binh (1-5)', fontsize=10)
ax1.set_title('So sanh chat luong cau sinh: ChatGPT vs Gemini', fontsize=12, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(axis='y', alpha=0.3)

# --- Heatmap naturalness by Rule ---
ax2 = axes[1]
pivot = df_eval.pivot_table(values='Naturalness', index='Source', columns='Rule', aggfunc='mean')
im = ax2.imshow(pivot.values, aspect='auto', cmap='YlGn', vmin=1, vmax=5)
ax2.set_xticks(range(len(pivot.columns)))
ax2.set_xticklabels(pivot.columns, fontsize=9)
ax2.set_yticks(range(len(pivot.index)))
ax2.set_yticklabels(pivot.index, fontsize=10)
for ri in range(pivot.shape[0]):
    for ci in range(pivot.shape[1]):
        val = pivot.values[ri, ci]
        if not np.isnan(val):
            ax2.text(ci, ri, f'{val:.1f}', ha='center', va='center', fontsize=11, fontweight='bold')
plt.colorbar(im, ax=ax2, label='Naturalness Score')
ax2.set_title('Heatmap Naturalness theo Rule', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('evaluation_charts.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Bieu do 2: Phan bo diem tung tieu chi
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
criteria = ['Naturalness','Meaning','Grammar']
palette  = ['#9C27B0','#00BCD4','#FF5722']

for i, (crit, color) in enumerate(zip(criteria, palette)):
    ax = axes[i]
    for src, hatch in zip(df_eval['Source'].unique(), ['','//']):
        subset = df_eval[df_eval['Source']==src][crit]
        ax.hist(subset, bins=[0.5,1.5,2.5,3.5,4.5,5.5], alpha=0.6,
                label=src, edgecolor='white', hatch=hatch)
    ax.set_title(f'{crit}\nMean={df_eval[crit].mean():.2f}', fontsize=11, fontweight='bold')
    ax.set_xlabel('Score (1-5)', fontsize=10)
    ax.set_ylabel('Count', fontsize=10)
    ax.set_xticks([1,2,3,4,5])
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Phan bo diem danh gia chat luong cau sinh', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('evaluation_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 1.5 – Lưu bộ dữ liệu

Ba file đầu ra theo yêu cầu đề bài:

| File | Nội dung | Số dòng |
|------|----------|---------|
| `Data.csv` | 50 cặp Q&A gốc + bản dịch VI + điểm BLEU + đánh giá Gemini | 50 |
| `DChatGPT.csv` | Câu sinh từ ChatGPT (50 ID × 5 luật) | 250 |
| `DGemini.csv` | Câu sinh từ Gemini (50 ID × 5 luật) | 250 |

In [ ]:
df_data.to_csv('Data.csv',     index=False, encoding='utf-8-sig')
df_chatgpt.to_csv('DChatGPT.csv', index=False, encoding='utf-8-sig')
df_gemini.to_csv('DGemini.csv',   index=False, encoding='utf-8-sig')

print('=== Da luu 3 file du lieu ===')
print(f'  Data.csv     : {len(df_data)} dong, {df_data.shape[1]} cot')
print(f'  DChatGPT.csv : {len(df_chatgpt)} dong')
print(f'  DGemini.csv  : {len(df_gemini)} dong')
print()
print('=' * 55)
print('TOM TAT KET QUA BAI 1')
print('=' * 55)
print(f'  Bo du lieu goc  : 50 cap Q&A (5 nhom x 10 cau)')
print(f'  Ngon ngu         : Tieng Anh + Tieng Viet (song ngu)')
print(f'  Dich bang        : Gemini (prompt hoc thuat)')
avg_eval = df_data[['Fluency','Adequacy','Grammar']].mean().mean()
print(f'  Danh gia dich    : TB {avg_eval:.2f}/5 (Fluency/Adequacy/Grammar)')
b_cgpt = bleu_cgpt[['BLEU1','BLEU4']].mean().round(3)
b_gem  = bleu_gem[['BLEU1','BLEU4']].mean().round(3)
print(f'  BLEU ChatGPT     : BLEU1={b_cgpt["BLEU1"]}, BLEU4={b_cgpt["BLEU4"]}')
print(f'  BLEU Gemini      : BLEU1={b_gem["BLEU1"]}, BLEU4={b_gem["BLEU4"]}')
print(f'  Tong cau sinh    : {len(df_chatgpt)} (ChatGPT) + {len(df_gemini)} (Gemini) = {len(df_chatgpt)+len(df_gemini)}')
print('=' * 55)